# Wordle 训练后端切换实践

本章把第 5～6 章介绍的配置落到 Wordle 训练中：先准备独立环境和训练资产，再核对启动参数，最后连续运行 3 个训练 step。所需适配已经包含在训练代码中，无需手工修改 Verl 或 TorchTitan-NPU 源码。

---


## 前置要求

为了完成本章实践，你应满足以下条件：

- 已完成第 5～6 章学习，理解后端切换范围、FSDP2 和 TND 变长注意力。
- 单机可见两张 Ascend NPU。
- CANN 9.0.0、Python 3.11 和 GCC 11 或更高版本可用。
- 工作区能够访问训练代码仓、Python 依赖源和 ModelScope。
- 宿主机具备安装依赖和保存模型权重所需的磁盘空间。

前四章生成的 `Qwen3-1.7B-Wordle-SFT` 与 Wordle parquet 已有时自动复用，缺失时自动准备。因此，全新工作区也可以从本章开始执行。

TorchTitan-NPU 使用单独的 uv 0.12.0 环境：Python 包安装在 `.venv`，固定并适配的上游源码保存在 `runtime_sources`。这两个目录不会改动初阶课程的 FSDP 环境。

---

## 章节目标

完成本章后，你将能够：

- 准备隔离的 TorchTitan-NPU 后端环境和 Wordle 训练资产。
- 通过 DRY_RUN 核对 FSDP2、TND 和原有强化学习语义配置。
- 启动三步训练并识别完整链路成功运行的关键日志。
- 说明三步运行能够确认哪些内容，以及性能对比还需要哪些条件。

---

## 本章内容

- [7.1 章节介绍](07.01_chapter_intro.ipynb)
- [7.2 准备后端环境与训练资产](07.02_prepare_backend.ipynb)
- [7.3 确认训练配置](07.03_confirm_configuration.ipynb)
- [7.4 运行三步训练](07.04_run_three_steps.ipynb)
- [7.5 章节练习](07.05_chapter_practice.ipynb)


---

## 实践路径

| 步骤 | Notebook | 完成标志 |
| --- | --- | --- |
| 1. 准备环境与资产 | 07.02 | 后端环境、SFT 模型与 Wordle parquet 均可用 |
| 2. 确认配置 | 07.03 | DRY_RUN 输出的启动命令包含 TorchTitan Engine、FSDP2、TND 和原有 RL 配置 |
| 3. 运行训练 | 07.04 | 保持初阶负载参数不变，完成 3 个训练 step |

launcher 默认将两张 NPU 组织为 DP shard 2、CP 1 的 FSDP2 拓扑。Actor 使用参数与优化器 offload，并在 forward 后 reshard；Ref 只使用参数 offload。TorchTitan 默认采用 BF16 参数与 FP32 reduction，同时启用激活重计算和 TND 变长注意力。完成三步运行后，可以沿日志确认从 rollout 到权重同步的各个阶段；收敛趋势需要更长时间的训练，性能收益则需要单独设计对比实验。
